# Crash!💥 Boom!💥 Bang! 💥🚗🚙💥🚗
**A SQL based exploratory data analysis of traffic accidents in Okinawa, Japan**

---   
# <span style="color:#0279f0;">Part 6: Key Findings & Recommendations</span>
---

We've looked at where and when accidents happened, who was involved, and the conditions at the time. What should we take away from all of this, and where should Okinawa look first?

**Analysis Tasks**
1. **Recheck the Key Numbers** – Pull the figures from DuckDB before drawing conclusions.
2. **Put the Findings Together** – Compare the places with the most crashes and the patterns behind fatal crashes.
3. **Decide What to Investigate Next** – Turn the findings into a short list of practical next steps.

##### 👩🏻‍💻 *<span style="color:#0279f0">Loading the checked figures...</span>*

In [1]:
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from executive_report.build_data import build_report_data

report = build_report_data()
display(pd.DataFrame([report["totals"]]))
display(pd.DataFrame(report["regions"]))
display(pd.DataFrame(report["lighting"]))
display(pd.DataFrame(report["types"]))
display(pd.DataFrame(report["holidays"]))

from IPython.display import Markdown

totals = report["totals"]
regions = {row["label"]: row["crashes"] for row in report["regions"]}
south_central = regions["South"] + regions["Central"]
top_city = report["municipalities"][0]
hours = pd.DataFrame(report["hours"]).groupby("hour")["crashes"].sum()
lighting = {row["code"]: row for row in report["lighting"]}
types = {row["label"]: row for row in report["types"]}
holidays = {row["code"]: row for row in report["holidays"]}

display(Markdown(f"""
### **6.1 The Big Picture**

The subset contains **{totals['crashes']:,} recorded casualty crashes with 2024 occurrence dates**.
There were **{totals['fatal_crashes']:,} fatal crashes**, **{totals['deaths']:,} deaths within 24 hours**,
and **{totals['injuries']:,} injuries**. The other {totals['injury_only_crashes']:,} crashes involved injuries only.

**Where?** Central and South together accounted for **{south_central:,} crashes
({south_central / totals['crashes']:.1%})**. **{top_city['label']}** had the highest municipal count
({top_city['crashes']:,}). **National Route 58** recorded **{report['route58']['crashes']:,} crashes**.

**When?** The **{hours.idxmax():02d}:00 hour** had the most recorded crashes ({hours.max():,}).
Full night recorded **{lighting[22]['fatal_crashes']} fatal crashes out of {lighting[22]['crashes']:,}**
({lighting[22]['fatal_crashes'] / lighting[22]['crashes']:.2%}), compared with
**{lighting[12]['fatal_crashes']} of {lighting[12]['crashes']:,}** in full daylight
({lighting[12]['fatal_crashes'] / lighting[12]['crashes']:.2%}).

**What?** Vehicle–vehicle crashes were the most common ({types['Vehicle–vehicle']['crashes']:,}).
Pedestrian–vehicle crashes contributed {types['Pedestrian–vehicle']['fatal_crashes']} fatal crashes.
Single-vehicle crashes accounted for {types['Single vehicle']['fatal_crashes']} fatal crashes
out of {types['Single vehicle']['crashes']}.

**What about holidays?** {holidays[1]['fatal_crashes']} of {holidays[1]['crashes']} holiday-coded
crashes were fatal. Part 4 compares these dates with regular Saturdays and Sundays and shows
intervals. The result is exploratory.
"""))

,crashes,fatal_crashes,injury_only_crashes,deaths,injuries,observed_days
0,2749,43,2706,44,3230,365


,label,crashes
0,Central,1270
1,South,1041
2,North,235
3,Yaeyama,105
4,Miyako,98


,code,label,crashes,fatal_crashes,deaths
0,11,First hour after sunrise,87,0,0
1,12,Full daylight,1728,18,18
2,13,Last hour before sunset,177,1,2
3,21,First hour after sunset,201,3,3
4,22,Full night,507,21,21
5,23,Last hour before sunrise,49,0,0


,label,crashes,fatal_crashes
0,Pedestrian–vehicle,469,17
1,Single vehicle,63,15
2,Vehicle–vehicle,2217,11


,code,crashes,fatal_crashes,deaths,observed_days
0,1,107,5,5,20
1,2,96,1,1,14
2,3,2546,37,38,331



### **6.1 The Big Picture**

The subset contains **2,749 recorded casualty crashes with 2024 occurrence dates**.
There were **43 fatal crashes**, **44 deaths within 24 hours**,
and **3,230 injuries**. The other 2,706 crashes involved injuries only.

**Where?** Central and South together accounted for **2,311 crashes
(84.1%)**. **Naha City** had the highest municipal count
(543). **National Route 58** recorded **322 crashes**.

**When?** The **17:00 hour** had the most recorded crashes (249).
Full night recorded **21 fatal crashes out of 507**
(4.14%), compared with
**18 of 1,728** in full daylight
(1.04%).

**What?** Vehicle–vehicle crashes were the most common (2,217).
Pedestrian–vehicle crashes contributed 17 fatal crashes.
Single-vehicle crashes accounted for 15 fatal crashes
out of 63.

**What about holidays?** 5 of 107 holiday-coded
crashes were fatal. Part 4 compares these dates with regular Saturdays and Sundays and shows
intervals. The result is exploratory.


### **6.2 What Should We Investigate First?**

| Focus | Where to start | What would help us decide? |
|---|---|---|
| **Route 58 and repeat crash locations** | Screen the Route 58 corridor and inspect junctions flagged in Part 3. | Match crash points to the road network, then check turning volumes and whether nearby grid cells represent the same junction. |
| **After-dark safety** | Review full-night fatal crashes for repeated locations and conditions. | Check site lighting, speed, and nighttime traffic volumes before choosing measures. |
| **Pedestrian crossings** | Review locations of fatal pedestrian–vehicle crashes and check nearby crossings. | Collect pedestrian volumes and document crossing layout, speeds, and visibility. |

---
### **6.3 What the Data Can Tell Us**

- This is one year of police-reported casualty crashes. The 2024 release contains 126 Okinawa records with earlier occurrence dates. Part 1 checks those dates and the crash keys independently. None of the excluded Okinawa rows was dated June 13, and the file leaves open whether June 13, 2024 had zero crashes or missing records.
- Crash counts show where incidents were recorded. Comparisons of road risk need a denominator such as vehicle trips, distance travelled, or pedestrian crossings.
- Deaths count people. Fatal crashes count incidents. A single crash can result in more than one death.
- The rounded coordinates in Part 3 are a first pass at spotting clusters. A junction can straddle multiple cells, and nearby 150 m neighbourhoods can overlap.

📊 [Open the executive report](../executive_report/index.html)